# Notebook para entendimiento y limpieza de las bases de datos raw



Este notebook construye la base del proyecto. Carga los estados financieros crudos
(carátula, balance y estado de resultados) reportados a la Superintendencia de
Sociedades para los años 2018–2024, los limpia (conserva solo el cierre anual y el
periodo actual, elimina duplicados y faltantes esenciales) y los une en un solo
panel con una fila por empresa y año.

Resultado: `data/processed/panel_2018_2024.csv`

Base de datos Carátula


In [1]:
import pandas as pd

ruta = "../data/raw/data_raw_2024/10000_Carátula.xlsx"

caratula = pd.read_excel(ruta)

# Tamaño y primeras 15 filas
print("Forma:", caratula.shape)
caratula.head(15)

Forma: (3116, 41)


,Punto de Entrada,Nombre Formulario,NIT,Fecha de Corte,El Máximo Órgano Social aprobó distribuir utilidades del ejercicio inmediatamente anterior,Valor de las utilidades decretadas en miles de pesos (ejercicio inmediatamente anterior),Razón social de la sociedad,Objeto social principal,Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU),Corte de cuentas según estatutos,...,Concepto del Revisor fiscal en su informe,Estos estados financieros presentan información reexpresada?,La información reexpresada corresponde a:,Reexpresión según normatividad que aplique,"De conformidad con lo dispuesto en el artículo 56 del CPACA, el suscrito autoriza a la Superintendencia de Sociedades para que, a partir de la fecha de presentación del informe, realice la notificación electrónica de los actos administrativos que correspondan.",E-mail de notificación electrónica,Dirección de notificación física,Departamento de notificación física,Ciudad (Municipio) de notificación física,"La Entidad posee inversiones en subsidiarias, asociadas y/o negocios conjuntos?"
0,Plenas-Individuales,Carátula,800171338,2024-03-31,NO,NaN,SOPLASCOL SAS,Fabricacion de envase y formas plasticas,C2229 - Fabricación de artículos de plástico n...,04. TRIMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,financiero@soplascol.com,NaN,NaN,NaN,NO
1,Plenas-Individuales,Carátula,800251163,2024-03-31,SI,667735254.0,OLEODUCTO CENTRAL S.A.,LA ACTIVIDAD PRINCIPAL DE OCENSA ES EL TRANSPO...,H4930 - Transporte por tuberías,04. TRIMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,notificaciones.judiciales@ocensa.com.co,NaN,NaN,NaN,NO
2,Plenas-Individuales,Carátula,900330496,2024-03-31,NO,NaN,INVERSIONES MOBEX S.A.S.,OSTENTAR LA TITULARIDAD DE ACCIONES DE LA SOCI...,G4799 - Otros tipos de comercio al por menor n...,04. TRIMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,mobex@siena.com.co,NaN,NaN,NaN,NO
3,Plenas-Individuales,Carátula,800068713,2024-06-30,SI,171482868.0,Oleoducto de Colombia S.A.,"La proyeccion,construccion y ejercicio de las ...",H4930 - Transporte por tuberías,02. SEMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,notificaciones.judiciales@oleoductodecolombia.com,NaN,NaN,NaN,NO
4,Plenas-Individuales,Carátula,800171338,2024-06-30,NO,NaN,SOPLASCOL SAS,Fabricacion de envase y formas plasticas,C2229 - Fabricación de artículos de plástico n...,04. TRIMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,financiero@soplascol.com,NaN,NaN,NaN,NO
5,Plenas-Individuales,Carátula,800251163,2024-06-30,SI,712493145.0,OLEODUCTO CENTRAL S.A.,LA ACTIVIDAD PRINCIPAL DE OCENSA ES EL TRANSPO...,H4930 - Transporte por tuberías,04. TRIMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,notificaciones.judiciales@ocensa.com.co,NaN,NaN,NaN,NO
6,Plenas-Individuales,Carátula,900269883,2024-06-30,SI,1550385.0,Zona Franca Metropolitana S.A.S. Usuario Operador,Realizar todas y exclusivamente las actividade...,M7020 - Actividades de consultoría de gestión,02. SEMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,jsalamanca@zfmetropolitana.com,NaN,NaN,NaN,NO
7,Plenas-Individuales,Carátula,900330496,2024-06-30,SI,17507605.0,INVERSIONES MOBEX S.A.S.,OSTENTAR LA TITULARIDAD DE ACCIONES DE LA SOCI...,G4799 - Otros tipos de comercio al por menor n...,04. TRIMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,mobex@siena.com.co,NaN,NaN,NaN,NO
8,Plenas-Individuales,Carátula,900529269,2024-06-30,SI,107578742.0,CFC GAS HOLDING SAS,La sociedad podrá desarrollar toda clase de ac...,M7010 - Actividades de administración empresarial,02. SEMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,cfcgas@corfi.com,NaN,NaN,NaN,NO
9,Plenas-Individuales,Carátula,900966852,2024-06-30,SI,14289033.0,CI REPSOL DUCTOS COLOMBIA S.A.S,La comercialización y venta de productos colom...,H4930 - Transporte por tuberías,02. SEMESTRAL,...,03. LIMPIO,NO,NaN,NaN,SI,notificacionesductos@ocensa.com.co,NaN,NaN,NaN,NO


Algunas se repiten varias veces en el archivo de un solo año, una por cada corte que reporta. Para el análisis se requiere una sola observación por empresa por año, y debe ser el cierre anual. De tal forma todas las empresas quedan medidas sobre el mismo periodo de doce meses y los ratios se vuelven comparables.

Las columnas que sirven de este archivo son: NIT, Razon social, Clasificación CIIU (sector), Estado actual y fecha de corte.

In [2]:
caratula.columns.tolist()

['Punto de Entrada',
 'Nombre Formulario',
 'NIT',
 'Fecha de Corte',
 'El Máximo Órgano Social aprobó distribuir utilidades del ejercicio inmediatamente anterior',
 'Valor de las utilidades decretadas en miles de pesos (ejercicio inmediatamente anterior)',
 'Razón social de la sociedad',
 'Objeto social principal',
 'Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU)',
 'Corte de cuentas según estatutos',
 'Fecha de constitución (Aaaa-Mm-Dd)',
 'Fecha de vencimiento (Aaaa-Mm-Dd)',
 'Estado actual',
 'Tipo societario',
 'La sociedad es',
 'Dirección de notificación judicial registrada en Cámara de Comercio',
 'Departamento de la dirección de notificación judicial',
 'Ciudad de la dirección de notificación judicial',
 'Dirección del domicilio',
 'Departamento de la dirección del domicilio',
 'Ciudad de la dirección del domicilio',
 'Teléfono del domicilio',
 'Celular corporativo',
 'E-mail de la sociedad',
 'Matricula mercantil número',
 'Domicilio casa matriz sucursal

In [3]:
print("Fechas de corte y cuántas filas de cada una:")
print(caratula["Fecha de Corte"].value_counts())

print("\nFilas totales:", len(caratula))
print("Empresas únicas (NIT):", caratula["NIT"].nunique())

Fechas de corte y cuántas filas de cada una:
Fecha de Corte
2024-12-31    3103
2024-06-30       8
2024-03-31       3
2024-09-30       2
Name: count, dtype: int64

Filas totales: 3116
Empresas únicas (NIT): 3104


De las 3116 filas 3103 son del cierre anual. Y hay 3104 empresas unicas. La mayoría de empresas reportó solo su cierre anual, lo que se necesita.

In [5]:
# Se convierte la columna a formato Fecha
caratula["Fecha de Corte"] = pd.to_datetime(caratula["Fecha de Corte"])

# Cierres de diciembre (mes 12)
caratula_anual = caratula[caratula["Fecha de Corte"].dt.month == 12].copy()

print("Filas diciembre:", len(caratula_anual))
print("NIT únicos:", caratula_anual["NIT"].nunique())

Filas diciembre: 3103
NIT únicos: 3103


In [7]:
# Nombre más corto de las columnas que se usaran
columnas = {
    "NIT": "nit",
    "Razón social de la sociedad": "razon_social",
    "Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU)": "ciiu",
    "Estado actual": "estado",
    "Fecha de Corte": "fecha_corte",
}

caratula_limpia = caratula_anual[list(columnas.keys())].rename(columns=columnas)

caratula_limpia.head()

,nit,razon_social,ciiu,estado,fecha_corte
13,800000946,Procter & Gamble Colombia LTDA,G4649 - Comercio al por mayor de otros utensil...,ACTIVA,2024-12-31
14,800001351,DANADOR SAS,K6499 - Otras actividades de servicio financie...,ACTIVA,2024-12-31
15,800003267,INTERGRAFIC DE OCCIDENTE SAS,C1811 - Actividades de impresión,ACTIVA,2024-12-31
16,800004599,COMERCIALIZADORA MARDEN LTDA,G4669 - Comercio al por mayor de otros product...,ACTIVA,2024-12-31
17,800005260,INDUSTRIAS INVERSIONES Y SERVICIOS DELRIO SAS,G4690 - Comercio al por mayor no especializado,ACTIVA,2024-12-31


In [8]:
caratula_limpia["estado"].value_counts()

estado
ACTIVA                         3004
ACUERDO DE REORGANIZACION        72
EN ETAPA PREOPERATIVA            14
ACUERDO DE REESTRUCTURACIÓN      12
CONCORDATO EN EJECUCIÓN           1
Name: count, dtype: int64

Base de datos de los Estados de situación financiera

In [9]:
import os
os.listdir("../data/raw/data_raw_2024")

['10000_Carátula.xlsx',
 '210030_Estado de situación financiera, corriente_no corriente.xlsx',
 '310030_Estado de resultado integral, resultado del periodo, por funcion de gasto.xlsx']

In [10]:
ruta_balance = "../data/raw/data_raw_2024/210030_Estado de situación financiera, corriente_no corriente.xlsx"
balance = pd.read_excel(ruta_balance)

print("Forma:", balance.shape)
balance.head(10)

Forma: (6232, 66)


,Punto de Entrada,Nombre Formulario,NIT,Fecha de Corte,Razón social de la sociedad,Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU),Tipo societario,Dirección de notificación judicial registrada en Cámara de Comercio,Departamento de la dirección del domicilio,Ciudad de la dirección del domicilio,...,Capital emitido (IssuedCapital),Prima de emisión (SharePremium),Acciones propias en cartera (TreasuryShares),Inversión suplementaria al capital asignado (InversionSuplementariaAlCapitalAsignado),Otras participaciones en el patrimonio (OtherEquityInterest),Superavit por revaluación (SuperavitPorRevaluacion),Otras reservas (OtherReserves),Ganancias acumuladas (RetainedEarnings),Patrimonio total (Equity),Total de patrimonio y pasivos (EquityAndLiabilities)
0,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",800171338,2024-03-31,SOPLASCOL SAS,C2229 - Fabricación de artículos de plástico n...,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,CR 54 5 C 96,BOGOTA D.C.,BOGOTA D.C.,...,3600000.0,NaN,NaN,NaN,NaN,11343798.0,600000.0,1.907788e+07,3.462168e+07,7.028852e+07
1,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",800171338,2024-03-31,SOPLASCOL SAS,C2229 - Fabricación de artículos de plástico n...,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,CR 54 5 C 96,BOGOTA D.C.,BOGOTA D.C.,...,3600000.0,NaN,NaN,NaN,NaN,11343798.0,600000.0,1.836591e+07,3.390970e+07,7.062742e+07
2,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",800251163,2024-03-31,OLEODUCTO CENTRAL S.A.,H4930 - Transporte por tuberías,01. SOCIEDAD ANÓNIMA,CRA 11 No 84-09 PISO 10,BOGOTA D.C.,BOGOTA D.C.,...,155309339.0,NaN,NaN,NaN,NaN,NaN,77654670.0,3.792536e+09,4.025500e+09,7.106465e+09
3,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",800251163,2024-03-31,OLEODUCTO CENTRAL S.A.,H4930 - Transporte por tuberías,01. SOCIEDAD ANÓNIMA,CRA 11 No 84-09 PISO 10,BOGOTA D.C.,BOGOTA D.C.,...,155309339.0,NaN,NaN,NaN,NaN,NaN,77654670.0,3.694325e+09,3.927289e+09,7.014520e+09
4,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",900330496,2024-03-31,INVERSIONES MOBEX S.A.S.,G4799 - Otros tipos de comercio al por menor n...,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,AVENIDA 3 OESTE # 10-51 EDIFICIO TORINO,VALLE,CALI-VALLE,...,1013249.0,83474630.0,NaN,NaN,NaN,NaN,238150462.0,3.007582e+07,3.527142e+08,3.888070e+08
5,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",900330496,2024-03-31,INVERSIONES MOBEX S.A.S.,G4799 - Otros tipos de comercio al por menor n...,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,AVENIDA 3 OESTE # 10-51 EDIFICIO TORINO,VALLE,CALI-VALLE,...,1013249.0,83474630.0,NaN,NaN,NaN,NaN,235043775.0,-2.129070e+05,3.193187e+08,3.548300e+08
6,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",800068713,2024-06-30,Oleoducto de Colombia S.A.,H4930 - Transporte por tuberías,01. SOCIEDAD ANÓNIMA,CALLE 113 NO 7 - 80 PISO 13,BOGOTA D.C.,BOGOTA D.C.,...,48595000.0,NaN,NaN,NaN,NaN,NaN,42368447.0,3.264752e+08,4.174386e+08,8.001801e+08
7,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",800068713,2024-06-30,Oleoducto de Colombia S.A.,H4930 - Transporte por tuberías,01. SOCIEDAD ANÓNIMA,CALLE 113 NO 7 - 80 PISO 13,BOGOTA D.C.,BOGOTA D.C.,...,48595000.0,NaN,NaN,NaN,NaN,NaN,42368447.0,3.251153e+08,4.160787e+08,8.459482e+08
8,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",800171338,2024-06-30,SOPLASCOL SAS,C2229 - Fabricación de artículos de plástico n...,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,CR 54 5 C 96,BOGOTA D.C.,BOGOTA D.C.,...,3600000.0,NaN,NaN,NaN,NaN,11343798.0,600000.0,1.915088e+07,3.469468e+07,6.875908e+07
9,Plenas-Individuales,"Estado de situación financiera, corriente/no c...",800171338,2024-06-30,SOPLASCOL SAS,C2229 - Fabricación de artículos de plástico n...,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,CR 54 5 C 96,BOGOTA D.C.,BOGOTA D.C.,...,3600000.0,NaN,NaN,NaN,NaN,11343798.0

La fila 0 y la fila 1 son la misma empresa en la misma fecha. lo mismo pasa con las filas 2-3, 4-5, etc. Cada empresa aprece duplicada pero los valores no son iguales. 
Esto se debe a que la norma obliga a la empresa a mostrar dos periodos (el actual y el del cierre anterior), por ello se ven dos filas por empresa.

Al igual que con la carátula, para este análisis es necesario contar unicamente con el corte de diciembre.

In [ ]:
# Ver todas las columnas
balance.columns.tolist()

['Punto de Entrada',
 'Nombre Formulario',
 'NIT',
 'Fecha de Corte',
 'Razón social de la sociedad',
 'Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU)',
 'Tipo societario',
 'Dirección de notificación judicial registrada en Cámara de Comercio',
 'Departamento de la dirección del domicilio',
 'Ciudad de la dirección del domicilio',
 'Periodo',
 'Efectivo y equivalentes al efectivo (CashAndCashEquivalents)',
 'Cuentas comerciales por cobrar y otras cuentas por cobrar corrientes (TradeAndOtherCurrentReceivables)',
 'Inventarios corrientes (Inventories)',
 'Activos por impuestos corrientes, corriente (CurrentTaxAssetsCurrent)',
 'Activos biológicos corrientes (CurrentBiologicalAssets)',
 'Otros activos financieros corrientes (OtherCurrentFinancialAssets)',
 'Otros activos no financieros corrientes (OtherCurrentNonfinancialAssets)',
 'Activos corrientes distintos al efectivo pignorados como garantía colateral para las que el receptor de transferencias tiene derecho por

La columna periodo parece ser justamente la que distingue entre periodo actual del comparativo, la que explica por qué cada empresa aparece dos veces.

En las columnas hay varias que son importantes para este análisis, sin embargo parece faltar pasivo corriente y total de pasivos, las que pudieron quedar en "..."

In [ ]:
# Inspección rapida para ver la columna periodo
balance["Periodo"].value_counts()

Periodo
Periodo Actual      3116
Periodo Anterior    3116
Name: count, dtype: int64

In [ ]:
# Intento de buscar columnas relacionadas con pasivos, activos o asivo (por si hay errores de tipeo)
for c in balance.columns:
    if "iabilities" in c or "(Assets)" in c or "asivo" in c.lower():
        print(c)

Total de activos (Assets)
Pasivos por impuestos corrientes, corriente (CurrentTaxLiabilitiesCurrent)
Otros pasivos financieros corrientes (OtherCurrentFinancialLiabilities)
Otros pasivos no financieros corrientes (OtherCurrentNonfinancialLiabilities)
Total de pasivos corrientes distintos de los pasivos incluidos en grupos de activos para su disposición clasificados como mantenidos para la venta (CurrentLiabilitiesOtherThanLiabilitiesIncludedInDisposalGroupsClassifiedAsHeldForSale)
Pasivos incluidos en grupos de activos para su disposición clasificados como mantenidos para la venta (LiabilitiesIncludedInDisposalGroupsClassifiedAsHeldForSale)
Pasivos corrientes totales (CurrentLiabilities)
Pasivo por impuestos diferidos (DeferredTaxLiabilities)
Pasivos por impuestos corrientes, no corriente (CurrentTaxLiabilitiesNoncurrent)
Otros pasivos financieros no corrientes (OtherNoncurrentFinancialLiabilities)
Otros pasivos no financieros no corrientes (OtherNoncurrentNonfinancialLiabilities)
Tota

Las cuentas que faltaban aparecieron y la columna periodo se divide exactamente en mitades (periodo actual y periodo anterior). 
Se filtrara todo por periodo actual que es el que compete.

In [15]:
balance["Fecha de Corte"] = pd.to_datetime(balance["Fecha de Corte"])

# Filtración de periodo actual y corte de diciembre 
balance_anual = balance[
    (balance["Periodo"] == "Periodo Actual") &
    (balance["Fecha de Corte"].dt.month == 12)
].copy()

print("Filas tras filtrar:", len(balance_anual))
print("NIT únicos:", balance_anual["NIT"].nunique())

Filas tras filtrar: 3103
NIT únicos: 3103


In [ ]:
# Columnas que se van a usar en el análisis con nombres más cortos
columnas_balance = {
    "NIT": "nit",
    "Activos corrientes totales (CurrentAssets)": "activo_corriente",
    "Total de activos (Assets)": "activo_total",
    "Pasivos corrientes totales (CurrentLiabilities)": "pasivo_corriente",
    "Total pasivos (Liabilities)": "pasivo_total",
    "Ganancias acumuladas (RetainedEarnings)": "ganancias_acumuladas",
    "Patrimonio total (Equity)": "patrimonio",
}

balance_limpio = balance_anual[list(columnas_balance.keys())].rename(columns=columnas_balance)

balance_limpio.head()

,nit,activo_corriente,activo_total,pasivo_corriente,pasivo_total,ganancias_acumuladas,patrimonio
26,800000946,599667064.0,734463544,458596797.0,468442814.0,143506232.0,266020730.0
28,800001351,229587570.0,230145825,45130846.0,45644746.0,181570739.0,184501079.0
30,800003267,39865440.0,89335239,39685736.0,64223188.0,18819546.0,25112051.0
32,800004599,16800366.0,27337946,18974674.0,21478065.0,-186320.0,5859881.0
34,800005260,43275665.0,52775435,12481273.0,13990748.0,5369228.0,38784687.0


3103 filas y 3103 NIT únicos, idéntico a la carátula.
Fila 26: activo total de 734463544	y si se suma pasivo total de 468442814.0 + patrimonio 266020730.0, da exactamente 734463544. Se cumple la identidad contable (Activo = Pasivo + Patrimonio). Eso confirma que las columnas están bien mapeadas y los datos cuadran.

Tercer y ultimo archivo: Estado de resultados integral

In [17]:
ruta_resultado = "../data/raw/data_raw_2024/310030_Estado de resultado integral, resultado del periodo, por funcion de gasto.xlsx"
resultado = pd.read_excel(ruta_resultado)

print("Forma:", resultado.shape)
print(resultado["Periodo"].value_counts())

Forma: (6232, 33)
Periodo
Periodo Actual      3116
Periodo Anterior    3116
Name: count, dtype: int64


Al igual que en el archivo anterior en este tambien viene duplicado el Periodo

In [18]:
# Buscar columnas relacionadas con ingresos, ganancias o pérdidas 
for c in resultado.columns:
    if "Revenue" in c or "ProfitLoss" in c or "OperatingActivities" in c or "ngreso" in c or "anancia" in c.lower():
        print(c)

Ingresos de actividades ordinarias (Revenue)
Ganancia bruta (GrossProfit)
Otros ingresos (OtherIncome)
Otras ganancias (pérdidas) (OtherGainsLosses)
Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)
Ganancias (pérdidas) que surgen de la baja en cuentas de activos financieros medidos al costo amortizado (GainLossArisingFromDerecognitionOfFinancialAssetsMeasuredAtAmortisedCost)
Ingresos financieros (FinanceIncome)
Deterioro de valor de ganancias y reversión de pérdidas por deterioro de valor (pérdidas por deterioro de valor) determinado de acuerdo con la NIIF 9 (ImpairmentLossImpairmentGainAndReversalOfImpairmentLossDeterminedInAccordanceWithIFRS9)
Ganancias (pérdidas) que surgen de diferencias entre el costo amortizado anterior y el valor razonable de activos financieros reclasificados de la categoría de medición costo amortizado a la categoría de medición de valor razonable con cambios en resultados (GainsLossesArisingFromDifferenceBetweenPreviousCarry

In [20]:
# Limpieza del archivo
resultado["Fecha de Corte"] = pd.to_datetime(resultado["Fecha de Corte"])

resultado_anual = resultado[
    (resultado["Periodo"] == "Periodo Actual") &
    (resultado["Fecha de Corte"].dt.month == 12)
].copy()

# Seleccionar y renombrar las tres cuentas que se necesiran para el análisis
columnas_resultado = {
    "NIT": "nit",
    "Ingresos de actividades ordinarias (Revenue)": "ingresos",
    "Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)": "utilidad_operacional",
    "Ganancia (pérdida) (ProfitLoss)": "utilidad_neta",
}

resultado_limpio = resultado_anual[list(columnas_resultado.keys())].rename(columns=columnas_resultado)

print("Filas:", len(resultado_limpio), "| NIT únicos:", resultado_limpio["nit"].nunique())
resultado_limpio.head()

Filas: 3103 | NIT únicos: 3103


,nit,ingresos,utilidad_operacional,utilidad_neta
26,800000946,1.266201e+09,74445595,53136652
28,800001351,1.912900e+04,7784,6929
30,800003267,7.158893e+07,11125933,3262505
32,800004599,2.771381e+07,18792,-69535
34,800005260,7.146925e+07,6284457,3864985


Se tienen tres tablas limpias, separadas y cada una con la columna NIT. Se necesita juntarlas para que cada empresa quede en una sola fila. Para lo cual se puede utilizar JOIN en SQL o merge de pandas.

In [21]:
# Unir carátula con balance por el NIT
datos_2024 = caratula_limpia.merge(balance_limpio, on="nit", how="inner")

# Se le pegar la tabla de resultados
datos_2024 = datos_2024.merge(resultado_limpio, on="nit", how="inner")

print("Forma final:", datos_2024.shape)
print("NIT únicos:", datos_2024["nit"].nunique())
datos_2024.head()

Forma final: (3103, 14)
NIT únicos: 3103


,nit,razon_social,ciiu,estado,fecha_corte,activo_corriente,activo_total,pasivo_corriente,pasivo_total,ganancias_acumuladas,patrimonio,ingresos,utilidad_operacional,utilidad_neta
0,800000946,Procter & Gamble Colombia LTDA,G4649 - Comercio al por mayor de otros utensil...,ACTIVA,2024-12-31,599667064.0,734463544,458596797.0,468442814.0,143506232.0,266020730.0,1.266201e+09,74445595,53136652
1,800001351,DANADOR SAS,K6499 - Otras actividades de servicio financie...,ACTIVA,2024-12-31,229587570.0,230145825,45130846.0,45644746.0,181570739.0,184501079.0,1.912900e+04,7784,6929
2,800003267,INTERGRAFIC DE OCCIDENTE SAS,C1811 - Actividades de impresión,ACTIVA,2024-12-31,39865440.0,89335239,39685736.0,64223188.0,18819546.0,25112051.0,7.158893e+07,11125933,3262505
3,800004599,COMERCIALIZADORA MARDEN LTDA,G4669 - Comercio al por mayor de otros product...,ACTIVA,2024-12-31,16800366.0,27337946,18974674.0,21478065.0,-186320.0,5859881.0,2.771381e+07,18792,-69535
4,800005260,INDUSTRIAS INVERSIONES Y SERVICIOS DELRIO SAS,G4690 - Comercio al por mayor no especializado,ACTIVA,2024-12-31,43275665.0,52775435,12481273.0,13990748.0,5369228.0,38784687.0,7.146925e+07,6284457,3864985


3103 filas y 14 columnas. El NIT más las 4 de identificación, las 6 del balance y las 3 de resultados. Queda completado el primer año entero.

En vez de repetir todo el proceso a mano (una por año), se va a empaquetar todo este proceso en una función y se apliucará en un bucle a los siete años, para luego apilarlos en una sola tabla grande.
Nota: Anio=año

In [24]:
import glob

def cargar_anio(anio):
    """Carga, limpia y une los tres estados financieros de un año.
    Devuelve un DataFrame con una fila por empresa."""

    carpeta = f"../data/raw/data_raw_{anio}"

    # Se busca cada archivo por su código
    ruta_caratula  = glob.glob(f"{carpeta}/10000*.xlsx")[0]
    ruta_balance   = glob.glob(f"{carpeta}/210030*.xlsx")[0]
    ruta_resultado = glob.glob(f"{carpeta}/310030*.xlsx")[0]

    # Carátula
    caratula = pd.read_excel(ruta_caratula)
    caratula["Fecha de Corte"] = pd.to_datetime(caratula["Fecha de Corte"])
    caratula = caratula[caratula["Fecha de Corte"].dt.month == 12].copy()
    cols_caratula = {
        "NIT": "nit",
        "Razón social de la sociedad": "razon_social",
        "Clasificación Industrial Internacional Uniforme Versión 4 A.C (CIIU)": "ciiu",
        "Estado actual": "estado",
        "Fecha de Corte": "fecha_corte",
    }
    caratula = caratula[list(cols_caratula.keys())].rename(columns=cols_caratula)

    # Balance
    balance = pd.read_excel(ruta_balance)
    balance["Fecha de Corte"] = pd.to_datetime(balance["Fecha de Corte"])
    balance = balance[
        (balance["Periodo"] == "Periodo Actual") &
        (balance["Fecha de Corte"].dt.month == 12)
    ].copy()
    cols_balance = {
        "NIT": "nit",
        "Activos corrientes totales (CurrentAssets)": "activo_corriente",
        "Total de activos (Assets)": "activo_total",
        "Pasivos corrientes totales (CurrentLiabilities)": "pasivo_corriente",
        "Total pasivos (Liabilities)": "pasivo_total",
        "Ganancias acumuladas (RetainedEarnings)": "ganancias_acumuladas",
        "Patrimonio total (Equity)": "patrimonio",
    }
    balance = balance[list(cols_balance.keys())].rename(columns=cols_balance)

    # Resultados
    resultado = pd.read_excel(ruta_resultado)
    resultado["Fecha de Corte"] = pd.to_datetime(resultado["Fecha de Corte"])
    resultado = resultado[
        (resultado["Periodo"] == "Periodo Actual") &
        (resultado["Fecha de Corte"].dt.month == 12)
    ].copy()
    cols_resultado = {
        "NIT": "nit",
        "Ingresos de actividades ordinarias (Revenue)": "ingresos",
        "Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)": "utilidad_operacional",
        "Ganancia (pérdida) (ProfitLoss)": "utilidad_neta",
    }
    resultado = resultado[list(cols_resultado.keys())].rename(columns=cols_resultado)

    # --- Unir las tres y marcar el año ---
    datos = caratula.merge(balance, on="nit", how="inner").merge(resultado, on="nit", how="inner")
    datos["anio"] = anio

    return datos

In [ ]:
# Prueba para comprobar con el año que ya se sabe qyue funciona (2024), si la función reproduce el mismo resultado (3103,15) queda validada
prueba = cargar_anio(2024)
print("Forma:", prueba.shape)
prueba.head()

Forma: (3103, 15)


,nit,razon_social,ciiu,estado,fecha_corte,activo_corriente,activo_total,pasivo_corriente,pasivo_total,ganancias_acumuladas,patrimonio,ingresos,utilidad_operacional,utilidad_neta,anio
0,800000946,Procter & Gamble Colombia LTDA,G4649 - Comercio al por mayor de otros utensil...,ACTIVA,2024-12-31,599667064.0,734463544,458596797.0,468442814.0,143506232.0,266020730.0,1.266201e+09,74445595,53136652,2024
1,800001351,DANADOR SAS,K6499 - Otras actividades de servicio financie...,ACTIVA,2024-12-31,229587570.0,230145825,45130846.0,45644746.0,181570739.0,184501079.0,1.912900e+04,7784,6929,2024
2,800003267,INTERGRAFIC DE OCCIDENTE SAS,C1811 - Actividades de impresión,ACTIVA,2024-12-31,39865440.0,89335239,39685736.0,64223188.0,18819546.0,25112051.0,7.158893e+07,11125933,3262505,2024
3,800004599,COMERCIALIZADORA MARDEN LTDA,G4669 - Comercio al por mayor de otros product...,ACTIVA,2024-12-31,16800366.0,27337946,18974674.0,21478065.0,-186320.0,5859881.0,2.771381e+07,18792,-69535,2024
4,800005260,INDUSTRIAS INVERSIONES Y SERVICIOS DELRIO SAS,G4690 - Comercio al por mayor no especializado,ACTIVA,2024-12-31,43275665.0,52775435,12481273.0,13990748.0,5369228.0,38784687.0,7.146925e+07,6284457,3864985,2024


La función quedo validada. Ya se puede aplicar a los siete años - BUCLE.

Al hacer el bucle dio KeyError para el año 2018, al parecer no se econtró la columna de CIIU. Posible error por cambio en los formularios de la Superintendencia.


In [ ]:
# Diagnostico
caratula_2018 = pd.read_excel(glob.glob("../data/raw/data_raw_2018/10000*.xlsx")[0])

for c in caratula_2018.columns:
    if "CIIU" in c or "lasificaci" in c.lower() or "ctividad" in c.lower():
        print(c)

Clasificación Industrial Internacional Uniforme Versión 4 A.C


Efectivamente es exactamente el mismo nombre la columna solo que en el 2024 se le agrego el (CIIU) al final mientras que en el 2018 no la tiene. Para solucionar esto, en lugar de buscar ek nombre exacto de la columna se busca por una parte estable de su nombre

In [34]:
# Función para encontrar palabra clave

def encontrar_columna(df, palabra_clave):
    """Devuelve el nombre de la primera columna que contiene la palabra clave."""
    for c in df.columns:
        if palabra_clave.lower() in c.lower():
            return c
    raise KeyError(f"No encontré ninguna columna con '{palabra_clave}'")

Función actualizada

In [30]:
def cargar_anio(anio):
    """Carga, limpia y une los tres estados financieros de un año.
    Devuelve un DataFrame con una fila por empresa."""

    carpeta = f"../data/raw/data_raw_{anio}"
    ruta_caratula  = glob.glob(f"{carpeta}/10000*.xlsx")[0]
    ruta_balance   = glob.glob(f"{carpeta}/210030*.xlsx")[0]
    ruta_resultado = glob.glob(f"{carpeta}/310030*.xlsx")[0]

    # Carátula
    caratula = pd.read_excel(ruta_caratula)
    caratula["Fecha de Corte"] = pd.to_datetime(caratula["Fecha de Corte"])
    caratula = caratula[caratula["Fecha de Corte"].dt.month == 12].copy()

    # Se busca ciiu por palabra clave
    col_ciiu = encontrar_columna(caratula, "Clasificación Industrial")
    cols_caratula = {
        "NIT": "nit",
        "Razón social de la sociedad": "razon_social",
        col_ciiu: "ciiu",
        "Estado actual": "estado",
        "Fecha de Corte": "fecha_corte",
    }
    caratula = caratula[list(cols_caratula.keys())].rename(columns=cols_caratula)

    # Balance
    balance = pd.read_excel(ruta_balance)
    balance["Fecha de Corte"] = pd.to_datetime(balance["Fecha de Corte"])
    balance = balance[
        (balance["Periodo"] == "Periodo Actual") &
        (balance["Fecha de Corte"].dt.month == 12)
    ].copy()
    cols_balance = {
        "NIT": "nit",
        "Activos corrientes totales (CurrentAssets)": "activo_corriente",
        "Total de activos (Assets)": "activo_total",
        "Pasivos corrientes totales (CurrentLiabilities)": "pasivo_corriente",
        "Total pasivos (Liabilities)": "pasivo_total",
        "Ganancias acumuladas (RetainedEarnings)": "ganancias_acumuladas",
        "Patrimonio total (Equity)": "patrimonio",
    }
    balance = balance[list(cols_balance.keys())].rename(columns=cols_balance)

    # Resultados
    resultado = pd.read_excel(ruta_resultado)
    resultado["Fecha de Corte"] = pd.to_datetime(resultado["Fecha de Corte"])
    resultado = resultado[
        (resultado["Periodo"] == "Periodo Actual") &
        (resultado["Fecha de Corte"].dt.month == 12)
    ].copy()
    cols_resultado = {
        "NIT": "nit",
        "Ingresos de actividades ordinarias (Revenue)": "ingresos",
        "Ganancia (pérdida) por actividades de operación (ProfitLossFromOperatingActivities)": "utilidad_operacional",
        "Ganancia (pérdida) (ProfitLoss)": "utilidad_neta",
    }
    resultado = resultado[list(cols_resultado.keys())].rename(columns=cols_resultado)

    datos = caratula.merge(balance, on="nit", how="inner").merge(resultado, on="nit", how="inner")
    datos["anio"] = anio
    return datos

In [32]:
#Bucle
tablas = []
for anio in anios:
    tabla = cargar_anio(anio)
    print(f"{anio}: {tabla.shape}")
    tablas.append(tabla)

panel = pd.concat(tablas, ignore_index=True)
print("\nPanel completo:", panel.shape)

2018: (2457, 15)
2019: (2686, 15)
2020: (2900, 15)
2021: (2895, 15)
2022: (3036, 15)
2023: (3120, 15)
2024: (3103, 15)

Panel completo: (20197, 15)


In [35]:
panel.to_csv("../data/processed/panel_2018_2024.csv", index=False)
print("Panel guardado en data/processed/")

Panel guardado en data/processed/


In [ ]:
#Revisar datos na
panel.isna().sum()

nit                        0
razon_social               1
ciiu                       0
estado                     0
fecha_corte                0
activo_corriente           4
activo_total               4
pasivo_corriente          64
pasivo_total               4
ganancias_acumuladas      12
patrimonio                 4
ingresos                1057
utilidad_operacional      23
utilidad_neta              4
anio                       0
dtype: int64

La mayoría de las columnas están casi perfectas: faltan entre 4 y 64 valores de más de 20.000 filas, una fracción mínima. Pero hay una que salta a la vista: ingresos tiene 1.057 faltantes.

In [37]:
panel.loc[panel["ingresos"].isna(), "ciiu"].str[0].value_counts()

ciiu
B    257
F    199
K    111
L    108
M     89
G     75
C     70
A     57
H     25
N     25
J     22
R      7
Q      6
I      2
D      2
E      1
U      1
Name: count, dtype: int64

Los sectores con ingresos faltantes: dominan B (minería): 257, F (construcción): 199, K (financiero): 111 y L (inmobiliario): 108.

Las cuentas del balance y las utilidades (faltan 4-64): son imprescindibles para el Z'' y los ratios principales, y no se puede inventar. Las filas que les falte alguna, las eliminamos. Son muy pocas.

Los ingresos se conservaran

In [38]:
columnas_esenciales = [
    "activo_corriente", "activo_total",
    "pasivo_corriente", "pasivo_total",
    "ganancias_acumuladas", "patrimonio",
    "utilidad_operacional", "utilidad_neta",
]

panel = panel.dropna(subset=columnas_esenciales)
print("Filas tras eliminar faltantes esenciales:", len(panel))

Filas tras eliminar faltantes esenciales: 20109


Para finalizar la fase de limpieza de datos y carga se guardan los datos ya procesados completamente.

In [39]:
panel.to_csv("../data/processed/panel_2018_2024.csv", index=False)